# app18/EdgeAI — treinamento da Random Forest que vai rodar **dentro** do ESP32Mesmo dataset do `app17-7`, mesma leitura do InfluxDB, mesmo corte por rodada. O que mudaé o modelo e o destino.| Arquivo gerado | Vai para | Serve para ||---|---|---|| `modelo_motor_rf.pkl` | seu computador (ou `CloudAI/api/`) | a floresta, em Python || `ModeloMotorScaler.hpp` | `EdgeAI/device/src/` | a média e o desvio de cada feature || `ModeloMotorRF.hpp` | `EdgeAI/device/src/` | a floresta, em `if`/`else` |São três formatos do mesmo modelo. Os dois `.hpp` saem do mesmo treino e viajam semprejuntos.

In [ ]:
# Versoes fixas: as mesmas do app17-7 e da CloudAI/api/, para o .pkl gerado aqui# abrir do outro lado. numpy e scikit-learn sao as que quebram o joblib.load se# divergirem. Os dois .hpp nao dependem de versao nenhuma: sao texto, C++ puro.## influxdb3-python le por SQL (Arrow Flight); pyarrow vem junto e e quem# converte o resultado em DataFrame. O micromlgen traduz a floresta em if/else.!pip install -q influxdb3-python pyarrow matplotlib "numpy==2.1.3" "pandas==2.2.3" "scikit-learn==1.6.1" "joblib==1.5.3" "micromlgen==1.1.28"

## 1) Conectar no InfluxDB CloudOs mesmos valores do notebook do `app17-7` — é o mesmo dataset, lido do mesmo lugar.O `bucket` do Node-RED é o `database` aqui.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport joblibfrom influxdb_client_3 import InfluxDBClient3from sklearn.ensemble import RandomForestClassifierfrom sklearn.pipeline import Pipelinefrom sklearn.preprocessing import StandardScalerINFLUX_URL    = "https://us-east-1-1.aws.cloud2.influxdata.com"INFLUX_TOKEN  = "SEU_TOKEN_INFLUX_CLOUD"INFLUX_BUCKET = "IoTSensores"          # o "database" no InfluxDB 3MEASUREMENT   = "vibracao_multiclasse"# A ORDEM desta lista e o contrato com o firmware: x[0] e mean_ax porque# mean_ax e a primeira coluna aqui. Mudar a ordem aqui sem mudar la faz o# modelo receber cada numero no lugar do outro.FEATURES = ["mean_ax", "mean_ay", "mean_az",            "std_ax", "std_ay", "std_az",            "std_mag", "p2p_mag"]CLASSES = ["operando", "inclinado_frente", "inclinado_tras", "anomalia"]client = InfluxDBClient3(host=INFLUX_URL, token=INFLUX_TOKEN, database=INFLUX_BUCKET)display(client.query("SHOW TABLES", language="sql").to_pandas())

## 2) Ler as janelasO mesmo `SELECT` do `app17-7`: as oito features, o rótulo e a rodada.

In [ ]:
colunas = ", ".join(f'"{f}"' for f in FEATURES)sql = f'''SELECT time, "label", "rodada", {colunas}FROM "{MEASUREMENT}"WHERE time >= now() - INTERVAL '30 days'ORDER BY time'''df = client.query(query=sql, language="sql").to_pandas()client.close()df = (df.query("label in @CLASSES")        .dropna(subset=FEATURES)        .sort_values("time")        .reset_index(drop=True))df["rodada"] = df["rodada"].astype(int)   # veio como tag, entao veio textoprint(f"{len(df)} janelas")print(df.groupby(["label", "rodada"]).size().to_string())

## 3) Split por rodada — a última rodada fica de fora do treinoIdêntico ao do notebook multiclasse do `app17-7`.

In [ ]:
rodada_teste = df["rodada"].max()treino = df[df["rodada"] != rodada_teste]teste  = df[df["rodada"] == rodada_teste]print(f"treino: {len(treino)} janelas | teste: rodada {rodada_teste}, {len(teste)} janelas")

## 4) Treinar a florestaQuatro escolhas, e cada uma aparece depois em algum arquivo gerado:- **`y` em texto**, como no `app17-7`. Assim o `predict()` devolve o nome da classe e o  `.pkl` funciona na API sem nenhuma adaptação. No ESP32 é diferente: lá o `predict()` do  micromlgen devolve o **índice** da classe, e quem dá nome a ele é o vetor  `NOMES_CLASSES` do firmware.- **`StandardScaler` no Pipeline.** Ele faz **padronização**: `(valor − média) / desvio`,  deixando cada feature com média 0 e desvio 1. A árvore *não precisa* disso — ela compara  uma feature por vez com um limiar, e mudar a escala não muda a ordem dos valores. Está  aqui por **simetria** — é o mesmo `Scaler::standardize()` do outro app embarcado da  disciplina. O que **não** pode é treinar com scaler e não padronizar no ESP32, ou o  contrário: os limiares do header ficariam numa escala e os dados em outra, e a predição  sai errada sem aviso nenhum.- **15 árvores, número ímpar.** No empate de votos o micromlgen escolhe o menor índice.  Número ímpar torna o empate raro.- **`max_features=None`.** Por padrão cada divisão sorteia só √8 ≈ 2 features, o que obriga  a árvore a crescer fundo para compensar as divisões ruins. Com 8 features, deixar a  árvore olhar todas custa quase nada e cada corte passa a ser o melhor corte: as árvores  ficam rasas, o header cabe na tela e você reconhece a física nos limiares.

In [ ]:
# Pipeline com os passos nomeados "scaler" e "clf", como no Colab do app30.modelo = Pipeline([    ("scaler", StandardScaler()),    ("clf",    RandomForestClassifier(n_estimators=15, max_features=None, random_state=42)),])modelo.fit(treino[FEATURES], treino["label"])scaler   = modelo.named_steps["scaler"]floresta = modelo.named_steps["clf"]# classes_ vem SEMPRE em ordem alfabetica, e e essa ordem que o indice do# micromlgen segue no ESP32: 0 = anomalia, 1 = inclinado_frente, e assim por diante.print("classes_ (a ordem dos indices no ESP32):", list(floresta.classes_))print("profundidade por arvore:", [a.tree_.max_depth for a in floresta.estimators_])

## 5) Avaliar

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, classification_report,                             ConfusionMatrixDisplay)ORDEM = list(floresta.classes_)y_pred = modelo.predict(teste[FEATURES])print("acuracia:", round(accuracy_score(teste["label"], y_pred), 3))print("f1_macro:", round(f1_score(teste["label"], y_pred, average="macro"), 3))print(classification_report(teste["label"], y_pred, labels=ORDEM, zero_division=0))ConfusionMatrixDisplay.from_predictions(teste["label"], y_pred, labels=ORDEM,                                        xticks_rotation=45, cmap="Blues")plt.tight_layout(); plt.show()

## 6) Quais features a floresta usouA floresta já traz essa conta pronta: cada divisão registra quanta impureza removeu, e aimportância é a soma disso por feature. Se `mean_ax` aparecer no topo, é a inclinação sendodetectada; se for `std_mag` ou `p2p_mag`, é a vibração.

In [ ]:
imp = pd.Series(floresta.feature_importances_, index=FEATURES).sort_values()imp.plot.barh(color="tab:green")plt.title("Importancia das features (Random Forest)")plt.tight_layout(); plt.show()print(imp.sort_values(ascending=False).round(3).to_string())

## 7) Salvar o modelo em `.pkl`O Pipeline inteiro, scaler e floresta juntos. É o formato que roda em Python — e, como o`y` foi treinado em texto, esse arquivo também roda na API do `CloudAI` sem adaptação.

In [ ]:
pkl_filename = "modelo_motor_rf.pkl"joblib.dump(modelo, pkl_filename)# Confere recarregando, que e o que a API faz na inicializacao.recarregado = joblib.load(pkl_filename)print("Predicao de 3 janelas de teste:", recarregado.predict(teste[FEATURES].head(3)))print(f"\nArquivo {pkl_filename} gerado.")

## 8) Exportar o scaler para C++O `.pkl` não serve no ESP32: lá não há Python. O scaler vira um header com a média e odesvio de cada feature, e uma função que faz a mesma conta.

In [ ]:
scaler_filename = "ModeloMotorScaler.hpp"means  = scaler.mean_.astype(float)scales = scaler.scale_.astype(float)conteudo = f"""#ifndef STANDARD_SCALER_HPP#define STANDARD_SCALER_HPP// Ordem: mean_ax, mean_ay, mean_az, std_ax, std_ay, std_az, std_mag, p2p_mag.namespace Scaler {{    const static float means[{len(means)}] = {{        {", ".join(f"{m:.10f}f" for m in means)}    }};    const static float scales[{len(scales)}] = {{        {", ".join(f"{s:.10f}f" for s in scales)}    }};    inline void standardize(const float* input, float* output) {{        for (int i = 0; i < {len(means)}; i++) {{            output[i] = (input[i] - means[i]) / scales[i];        }}    }}}}#endif"""with open(scaler_filename, "w") as f:    f.write(conteudo)print(conteudo)print(f"\nArquivo {scaler_filename} gerado.")

## 9) Exportar a floresta com o micromlgenUma chamada. O `port()` percorre as 15 árvores e escreve cada uma como uma sequência de`if`/`else` — é literalmente isso que uma árvore de decisão é.No ESP32, o `predict()` desse header devolve o **índice** da classe. O vetor impresso nofim da célula é o que traduz esse número de volta em nome, e vai para o `.cpp`.

In [ ]:
from micromlgen import portc_code = port(floresta)print(c_code)micromlgen_filename = "ModeloMotorRF.hpp"with open(micromlgen_filename, 'w') as f:    f.write(c_code)print(f"\nArquivo {micromlgen_filename} gerado.")print("\nCole esta linha no .cpp do firmware:")print('const char* NOMES_CLASSES[4] = { '      + ", ".join(f'"{c}"' for c in floresta.classes_) + ' };')

## 10) BaixarOs dois `.hpp` vão para `EdgeAI/device/src/`, por cima dos sintéticos que vieram noprojeto. O `.pkl` é seu — e, se quiser rodar esta floresta na API em vez da rede neural,coloque-o em `CloudAI/api/` e suba a API apontando para ele:`MODELO_ARQUIVO=modelo_motor_rf.pkl uvicorn service_app:app --host 0.0.0.0 --port 8000`.

In [ ]:
import zipfile# Um zip so: os tres arquivos precisam viajar juntos.with zipfile.ZipFile("modelo_motor_rf.zip", "w") as pacote:    pacote.write(pkl_filename)    pacote.write(scaler_filename)    pacote.write(micromlgen_filename)try:    from google.colab import files    files.download("modelo_motor_rf.zip")except Exception:    print("modelo_motor_rf.zip gerado na pasta atual.")